# v10 — asymmetric recipe on REAL model KV (the honest test, Colab T4)

The synthetic ablation REFUTED the asymmetric recipe — on i.i.d. Gaussian KV, per-channel-K / per-token-V is *worse* than block16, because Gaussian noise has no outlier structure to exploit. per-channel-K / per-token-V (KIVI/KVQuant/KVTuner) only pay when REAL outliers exist: K has a few huge **channels**, V has huge **tokens** (attention sinks / BOS). This notebook captures REAL K,V from GPT-2 on real text and reruns the K×V matrix where that structure actually lives — the genuine test of the recipe. Still pure fake-quant (no kernel/build). FP8 here is **per-tensor** (the v9 recipe); the sharp question: does **fine-grained FP4 (per-channel-K / block16) approach or beat coarse per-tensor FP8** once real outliers punish the coarse scale?

## 0. Dependencies + GPU (venv-safe) + transformers

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes.
pip('ninja', 'pytest', 'numpy', 'transformers')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

# torch.float8_e4m3fn must exist (>=2.1) — v9 stores the KV cache as E4M3 bytes.
assert hasattr(torch, 'float8_e4m3fn'), 'this torch lacks float8_e4m3fn; upgrade torch (>=2.1)'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Capture REAL K,V from GPT-2 on real text (per-layer)

In [ ]:
# Hook each block's fused QKV projection (c_attn), run one forward on a real passage, and reshape the
# captured q,k,v to [B, H, N, d]. GPT-2 is MHA (H_kv=H, G=1) with NO RoPE, so the raw q,k are exactly
# what attention consumes (scale 1/sqrt(d) applied inside). Outlier magnitude grows with depth -> we
# keep early/mid/late layers.
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').cuda().eval()
cfg = model.config
H, d, L = cfg.n_head, cfg.n_embd // cfg.n_head, cfg.n_layer
print(f'GPT-2: layers={L} heads={H} d_head={d} (MHA, G=1)')

PASSAGE = "The mitochondrion is a double-membrane-bound organelle found in most eukaryotic cells. Mitochondria generate most of the cell's supply of adenosine triphosphate, used as a source of chemical energy. They were first discovered in the nineteenth century, and the term mitochondrion was coined by Carl Benda in 1898. The organelle is composed of an outer membrane, an inner membrane, an intermembrane space, and the matrix. The inner membrane is folded into structures called cristae, which increase the surface area available for chemical reactions. Mitochondria contain their own genome, a small circular molecule of DNA distinct from the nuclear genome, a feature consistent with the endosymbiotic theory, which proposes that mitochondria descended from free-living bacteria that were engulfed by an ancestral eukaryotic cell. The number of mitochondria in a cell varies widely by organism, tissue, and cell type; a red blood cell has none, whereas a liver cell can contain more than two thousand. The organelle is bounded by two membranes, each a phospholipid bilayer with embedded proteins, and the two membranes have very different properties. Because of this, the mitochondrion has five distinct compartments: the outer membrane, the intermembrane space, the inner membrane, the cristae space formed by infoldings of the inner membrane, and the matrix, which is the space within the inner membrane. Mitochondria are involved in many other processes besides the production of energy, including signaling, cellular differentiation, and cell death, as well as maintaining control of the cell cycle and cell growth. Mitochondrial biogenesis is in turn temporally coordinated with these cellular processes. Several characteristics make mitochondria unique. The number of mitochondria in a cell can vary widely by organism, tissue, and cell type. A mature red blood cell has no mitochondria, whereas a liver cell can have more than two thousand. Mitochondria are commonly between point seven five and three micrometers in diameter but vary considerably in size and structure. Unless specifically stained, they are not visible under an ordinary microscope. "
ids = tok(PASSAGE, return_tensors='pt').input_ids[:, :512].cuda()
print('sequence length:', ids.shape[1])

cap = {}
def mk_hook(i):
    def hook(m, inp, out):                      # out = c_attn(x): [B, N, 3*n_embd]
        q, k, v = out.split(cfg.n_embd, dim=2)  # GPT2Attention split order is q,k,v
        B, N, _ = q.shape
        shp = lambda x: x.view(B, N, H, d).permute(0, 2, 1, 3).contiguous()  # [B,H,N,d]
        cap[i] = (shp(q).float(), shp(k).float(), shp(v).float())
    return hook
handles = [model.transformer.h[i].attn.c_attn.register_forward_hook(mk_hook(i)) for i in range(L)]
with torch.no_grad():
    model(ids)
for hd in handles:
    hd.remove()
LAYERS = [2, 6, 11]    # early / mid / late
print('captured layers', sorted(cap), '| K shape', tuple(cap[0][1].shape))

## 3. Outlier diagnostic — PROVE the structure exists (vs Gaussian)

In [ ]:
# The recipe's whole premise. Quantify per-CHANNEL outlier-ness of K (max|.|/mean|.| over tokens, per
# channel -> a few channels should spike) and per-TOKEN outlier-ness of V (per-token L2 norm -> sink
# tokens should spike). Gaussian baseline: max/mean ~ 3-4. Real K/V should show channels/tokens FAR
# above that, which is exactly what per-channel-K / per-token-V can capture and block16 cannot.
import torch
print(f'{"layer":>5} | {"K chan max/mean (top3)":>26} | {"V tok norm max/median":>22}')
for i in LAYERS:
    q, k, v = cap[i]
    kk = k[0]                                   # [H, N, d]
    chan = kk.abs().amax(dim=1) / kk.abs().mean(dim=1).clamp_min(1e-9)   # [H, d] per-channel max/mean
    top3 = chan.flatten().topk(3).values.tolist()
    vv = v[0]                                    # [H, N, d]
    tnorm = vv.norm(dim=-1)                      # [H, N] per-token norm
    ratio = (tnorm.amax(dim=1) / tnorm.median(dim=1).values.clamp_min(1e-9)).mean().item()
    print(f'{i:>5} | {" ".join(f"{x:6.1f}" for x in top3):>26} | {ratio:22.2f}')
g = torch.randn(8, 512, 64, device='cuda')
gchan = (g.abs().amax(dim=1) / g.abs().mean(dim=1)).flatten().topk(3).values.tolist()
gtok = (g.norm(dim=-1).amax(dim=1) / g.norm(dim=-1).median(dim=1).values).mean().item()
print(f'{"GAUSS":>5} | {" ".join(f"{x:6.1f}" for x in gchan):>26} | {gtok:22.2f}   <- the no-structure baseline')
print('\nIf real channels/tokens spike far above the Gaussian row, per-channel-K/per-token-V has signal.')

## 4. The K×V matrix on REAL KV — does asymmetric beat block16 now?

In [ ]:
# Same matrix as the synthetic notebook, but on REAL captured KV (decode: last-token query). FP8 floor
# is per-tensor E4M3 (the v9 recipe). A trailing * = met/beat that FP8 floor.
import statistics as st, torch
from fa_kernels.nvfp4_recipes import attn_rmse, fp8_attn_rmse, GRANULARITIES

def run_real(i):
    q, k, v = cap[i]
    qd = q[:, :, -1:, :].contiguous()           # decode: last-token query [B,H,1,d]
    fp8 = fp8_attn_rmse(qd, k, v)
    print(f'=== layer {i} | FP8(per-tensor) floor = {fp8:.3e} ===')
    print(f"{'K\\V':>8} | " + ' | '.join(f'{vg:>9}' for vg in GRANULARITIES))
    best = (1e9, None)
    for kg in GRANULARITIES:
        cs = []
        for vg in GRANULARITIES:
            m = attn_rmse(qd, k, v, k_gran=kg, v_gran=vg)
            if m < best[0]: best = (m, (kg, vg))
            cs.append(f'{m:.2e}' + ('*' if m <= fp8 else ' '))
        print(f'{kg:>8} | ' + ' | '.join(f'{c:>9}' for c in cs))
    print(f'  best NVFP4 cell: {best[1]} = {best[0]:.3e} ({best[0]/fp8:.2f}x FP8)')
    return best, fp8

for i in LAYERS:
    run_real(i); print()
print('VERDICT to read off: is the best cell (K=channel,*) and does it now beat block16 / approach FP8?')